# k-Nearest Neighbor (kNN) exercise

The kNN classifier consists of two stages:

- During training, the classifier takes the training data and simply remembers it
- During testing, kNN classifies every test image by comparing to all training images and transfering the labels of the k most similar training examples
- The value of k is cross-validated

In this exercise you will implement these steps and understand the basic Image Classification pipeline, cross-validation, and gain proficiency in writing efficient, vectorized code.

In [ ]:
# 做一些准备性的工作，print_function目的是为了要兼容py2，py3的输出。
from __future__ import print_function
import sys
sys.path.append('/kaggle/input/ms326-knn')
import random
import numpy as np
import matplotlib.pyplot as plt

# 让图片生成在notebook中而不是生成新的窗口
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'


#作用：在调试的过程中，如果代码发生更新，实现ipython中引用的模块也能自动更新。
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

In [ ]:
# Load the raw Fashion_mnist data.
import os
import pandas
with open("../input/fashionmnist/fashion-mnist_test.csv", "r") as f:
    test_data = pandas.read_csv(f).values
with open("../input/fashionmnist/fashion-mnist_train.csv", "r") as f:
    train_data = pandas.read_csv(f).values
    
X_train = train_data[:, 1:].astype(np.float32)
y_train = train_data[:, 0]

X_test = train_data[:, 1:].astype(np.float32)
y_test = train_data[:, 0]

# As a sanity check, we print out the size of the training and test data.
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

# Fashion-MNIST
  Fashion-MNIST 是⼀个10 类服饰分类数据集。Fashion-MNIST 中⼀共包括了10 个类别，分别为：t-shirt（T 恤）、trouser（裤⼦）、pullover（套衫）、dress（连⾐裙）、coat（外套）、sandal（凉鞋）、shirt（衬衫）、sneaker（运动鞋）、bag（包）和ankle boot（短靴）。 我们可以通过可视化来很好的了解数据集

In [ ]:
# np.flatnonzero 矩阵扁平化后返回非零元素的位置
# numpy.random.choice(a, size=None, replace=True, p=None) 随机选取a中的值
classes = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.subplots_adjust(left=0, bottom=None, right=None, top=None,
                wspace=0.1, hspace=0.1)
        plt.imshow(X_train[idx].reshape((28,28)),cmap=plt.cm.gray)
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
# Subsample the data for more efficient code execution in this exercise
num_training = 5000
mask = list(range(num_training))
X_train = X_train[mask]
y_train = y_train[mask]

num_test = 500
mask = list(range(num_test))
X_test = X_test[mask]
y_test = y_test[mask]

In [ ]:
# Reshape the image data into rows
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_train = np.array(X_train, dtype = np.int32)
X_test = np.reshape(X_test, (X_test.shape[0], -1))
X_test = np.array(X_test, dtype = np.int32)
print(X_train.shape, X_test.shape)

In [ ]:
from k_nearest_neighbor import KNearestNeighbor

# Create a kNN classifier instance. 
# Remember that training a kNN classifier is a noop: 
# the Classifier simply remembers the data and does no further processing 
classifier = KNearestNeighbor()
classifier.train(X_train, y_train)
# While encounter 'No module named 'past'' : run pip install future

We would now like to classify the test data with the kNN classifier. Recall that we can break down this process into two steps: 

1. First we must compute the distances between all test examples and all train examples. 
2. Given these distances, for each test example we find the k nearest examples and have them vote for the label

Lets begin with computing the distance matrix between all training and test examples. For example, if there are **Ntr** training examples and **Nte** test examples, this stage should result in a **Nte x Ntr** matrix where each element (i,j) is the distance between the i-th test and j-th train example.

**Note: For the three distance computations that we require you to implement in this notebook, you may not use the np.linalg.norm() function that numpy provides.**

First, open `MS326/classifiers/k_nearest_neighbor.py` and implement the function `compute_distances_two_loops` that uses a (very inefficient) double loop over all pairs of (test, train) examples and computes the distance matrix one element at a time.

In [ ]:
# Open MS326/classifiers/k_nearest_neighbor.py and implement
# compute_distances_two_loops.
def compute_distances_two_loops(self, X):
    num_test = X.shape[0]
    num_train = self.X_train.shape[0]
    dists = np.zeros((num_test, num_train))
    for i in xrange(num_test):
        for j in xrange(num_train):
            dists[i][j] = np.sqrt(np.sum(np.square(X[i] - self.X_train[j])))
    return dists


# Test your implementation:
dists = classifier.compute_distances_two_loops(X_test)
print(dists.shape)

In [ ]:
# We can visualize the distance matrix: each row is a single test example and
# its distances to training examples
plt.imshow(dists, interpolation='none')
plt.show()

**Inline Question #1:** Notice the structured patterns in the distance matrix, where some rows or columns are visible brighter. (Note that with the default color scheme black indicates low distances while white indicates high distances.)

- What in the data is the cause behind the distinctly bright rows?
- What causes the columns?

**Your Answer**: 
*1.If the ith test data is similar to a large number of train data, the ith row would be black. Otherwise,the ith row would be white.*
*2 .If the jth train data is similar to a large number of test data, the jth column would be black. Otherwise,the jth column would be white.*




In [ ]:
# Now implement the function predict_labels and run the code below:
# We use k = 1 (which is Nearest Neighbor).
y_test_pred = classifier.predict_labels(dists, k=1)

# Compute and print the fraction of correctly predicted examples
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

You should expect to see approximately `80%` accuracy. Now lets try out a larger `k`, say `k = 5`:

In [ ]:
y_test_pred = classifier.predict_labels(dists, k=5)
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

You should expect to see a slightly better performance than with `k = 1`.

**Inline Question 2**

We can also use other distance metrics such as L1 distance.
For pixel values $p_{ij}^{(k)}$ at location $(i,j)$ of some image $I_k$, 

the mean $\mu$ across all pixels over all images is $$\mu=\frac{1}{nhw}\sum_{k=1}^n\sum_{i=1}^{h}\sum_{j=1}^{w}p_{ij}^{(k)}$$
And the pixel-wise mean $\mu_{ij}$ across all images is 
$$\mu_{ij}=\frac{1}{n}\sum_{k=1}^np_{ij}^{(k)}.$$
The general standard deviation $\sigma$ and pixel-wise standard deviation $\sigma_{ij}$ is defined similarly.

Which of the following preprocessing steps will not change the performance of a Nearest Neighbor classifier that uses L1 distance? Select all that apply.
1. Subtracting the mean $\mu$ ($\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu$.)
2. Subtracting the per pixel mean $\mu_{ij}$  ($\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu_{ij}$.)
3. Subtracting the mean $\mu$ and dividing by the standard deviation $\sigma$.
4. Subtracting the pixel-wise mean $\mu_{ij}$ and dividing by the pixel-wise standard deviation $\sigma_{ij}$.
5. Rotating the coordinate axes of the data.

$\color{blue}{\textit Your Answer:}$ 
1,3


$\color{blue}{\textit Your Explanation:}$
1. Each pixel subtracts the same average value, so the $L1$ loss will not change.
2. Each pixel subtracts the different average value, so the $L1$ loss will change.
3. Like 1, $\|(x1 - \mu)/\sigma - (x2 - \mu)/\sigma\| = \|x1 - x2\|/\sigma$, The ratio of distance is the same, so the performance of $L1$ is not affected.
4. Like 2,so the $L1$ loss will change.
5. Rotating the coordinate axes of the data will influence the $L1$ loss, however $L2$ loss will not change. Just like $(0,0)$,$(2,2)$. If we rotate the coordinate axes to make them $(0,2\sqrt{2})$,$(0,0)$


In [ ]:
# Now lets speed up distance matrix computation by using partial vectorization
# with one loop. Implement the function compute_distances_one_loop and run the
# code below:

# python Broadcasting机制
def compute_distances_one_loop(self, X):
    num_test = X.shape[0]
    num_train = self.X_train.shape[0]
    dists = np.zeros((num_test, num_train))
    for i in xrange(num_test):
        dists[i] = np.sqrt(np.sum(np.square(self.X_train - X[i]), axis = 1))
    return dists

dists_one = classifier.compute_distances_one_loop(X_test)

# 我们可以用 Frobenius 范数去判断我们是否求解正确了. 
# In case you haven't seen it before, the Frobenius norm of two matrices is the square
# root of the squared sum of differences of all elements; in other words, reshape
# the matrices into vectors and compute the Euclidean distance between them.
difference = np.linalg.norm(dists - dists_one, ord='fro')
print('Difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')

**Compute_distances_no_loops**

$原理：假设X是测试集(m*d),训练集Y是(n*d),其中m是测试数据数量，n是训练数据数量，d是维度。计算两者公式如下：$
\begin{equation}
\sum_{i=0}^{m} \sum_{j=0}^{n}\sqrt{(P_i - C_j)^2} = \sum_{i=0}^{m} \sum_{j=0}^{n}\sqrt{\|P_i\|^2 + \|C_j\|^2 - 2 * p_ic_j}
\end{equation}


In [ ]:
# Now implement the fully vectorized version inside compute_distances_no_loops
# and run the code
def compute_distances_no_loops(self, X):
    num_test = X.shape[0]
    num_train = self.X_train.shape[0]
    dists = np.zeros((num_test, num_train)) 
    #########################################################################
    # HINT: Try to formulate the l2 distance using matrix multiplication    #
    #       and two broadcast sums                                          #
    # Attention: square(x1-x2) = square(x1) + square(x2) - 2 * x1 * x2      #
    #########################################################################
    x1_x2 = np.dot(X, self.X_train.T)  # num_test * num_train
    test_square = np.sum(np.square(X), axis=1).reshape(-1,1) # num_test * 1
    train_square = np.sum(np.square(self.X_train.T), axis=0).reshape(1,-1) # 1 * num_trian
    dists = np.sqrt(-2 * x1_x2 + test_square + train_square) #broadcast
    return dists


dists_two = classifier.compute_distances_no_loops(X_test)
# check that the distance matrix agrees with the one we computed before:
difference = np.linalg.norm(dists - dists_two, ord='fro')
print('Difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')

In [ ]:
# Let's compare how fast the implementations are
def time_function(f, *args):
    """
    Call a function f with args and return the time (in seconds) that it took to execute.
    """
    import time
    tic = time.time()
    f(*args)
    toc = time.time()
    return toc - tic

two_loop_time = time_function(classifier.compute_distances_two_loops, X_test)
print('Two loop version took %f seconds' % two_loop_time)

one_loop_time = time_function(classifier.compute_distances_one_loop, X_test)
print('One loop version took %f seconds' % one_loop_time)

no_loop_time = time_function(classifier.compute_distances_no_loops, X_test)
print('No loop version took %f seconds' % no_loop_time)

# you should see significantly faster performance with the fully vectorized implementation

### Cross-validation

We have implemented the k-Nearest Neighbor classifier but we set the value k = 5 arbitrarily. We will now determine the best value of this hyperparameter with cross-validation.

In [ ]:
num_folds = 5
k_choices = [1, 3, 5, 8, 10, 12, 15, 20, 50, 100]

X_train_folds = []
y_train_folds = []
################################################################################                                                                       #
# Split up the training data into folds. After splitting, X_train_folds and    #
# y_train_folds should each be lists of length num_folds, where                #
# y_train_folds[i] is the label vector for the points in X_train_folds[i].     #
# Hint: Look up the numpy array_split function.                                #
################################################################################
y_train_ = y_train.reshape(-1, 1)
X_train_folds , y_train_folds = np.array_split(X_train, num_folds), np.array_split(y_train_, num_folds)

# A dictionary holding the accuracies for different values of k that we find
# when running cross-validation. After running cross-validation,
# k_to_accuracies[k] should be a list of length num_folds giving the different
# accuracy values that we found when using that value of k.
k_to_accuracies = {}

################################################################################
# Perform k-fold cross validation to find the best value of k. For each        #
# possible value of k, run the k-nearest-neighbor algorithm num_folds times,   #
# where in each case you use all but one of the folds as training data and the #
# last fold as a validation set. Store the accuracies for all fold and all     #
# values of k in the k_to_accuracies dictionary.                               #
################################################################################
for k in k_choices:
    k_to_accuracies.setdefault(k, [])

for i in range(num_folds):
    classifier = KNearestNeighbor()
    X_val_train = np.vstack(X_train_folds[0:i] + X_train_folds[i+1:])
    y_val_train = np.vstack(y_train_folds[0:i] + y_train_folds[i+1:])[:,0]
    classifier.train(X_val_train, y_val_train)
    for k in k_choices:
        y_val_pred = classifier.predict(X_train_folds[i], k=k)
        num_correct = np.sum(y_val_pred == y_train_folds[i][:,0])
        accuracy = float(num_correct) / len(y_val_pred)
        k_to_accuracies[k] = k_to_accuracies[k] + [accuracy]
    
# Print out the computed accuracies
for k in sorted(k_to_accuracies):
    for accuracy in k_to_accuracies[k]:
        print('k = %d, accuracy = %f' % (k, accuracy))

In [ ]:
# plot the raw observations
for k in k_choices:
    accuracies = k_to_accuracies[k]
    plt.scatter([k] * len(accuracies), accuracies)

# plot the trend line with error bars that correspond to standard deviation
accuracies_mean = np.array([np.mean(v) for k,v in sorted(k_to_accuracies.items())])
accuracies_std = np.array([np.std(v) for k,v in sorted(k_to_accuracies.items())])
plt.errorbar(k_choices, accuracies_mean, yerr=accuracies_std)
plt.title('Cross-validation on k')
plt.xlabel('k')
plt.ylabel('Cross-validation accuracy')
plt.show()

In [ ]:
# Based on the cross-validation results above, choose the best value for k,   
# retrain the classifier using all the training data, and test it on the test
# data. You should be able to get above 83% accuracy on the test data.
best_k = 15

classifier = KNearestNeighbor()
classifier.train(X_train, y_train)
y_test_pred = classifier.predict(X_test, k=best_k)

# Compute and display the accuracy
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

**Inline Question 3**

Which of the following statements about $k$-Nearest Neighbor ($k$-NN) are true in a classification setting, and for all $k$? Select all that apply.
1. The decision boundary of the k-NN classifier is linear.
2. The training error of a 1-NN will always be lower than that of 5-NN.
3. The test error of a 1-NN will always be lower than that of a 5-NN.
4. The time needed to classify a test example with the k-NN classifier grows with the size of the training set.
5. None of the above.

$\color{blue}{\textit Your Answer:}$ 2，4


$\color{blue}{\textit Your Explanation:}$ 
1. The decision boundary is not linear because it calculates the distance to the vertex. Suppose a class has a vertex x in the center and the other class has lots of vertexs with distance 2 to vertex. And we use 1-NN. We can see the boundary is nearly a circle with the radius 1.
2. When k=1, it means that only the nearest point can be judged, so there is no error in training. When k=5, there will be different training errors according to the rule of vote.
3. It is decided by the data distribution and can be easily constructed in small dataset.
4. Because the method should compare each test data with each training set. The larger the training set, the larger the distance matrix, the bigger the running time.




